# TLMAC Integrated Flow

Full pipeline from N2UQ quantised ResNet-18 to FPGA LUT INIT files:

1. Load N2UQ quantised weights from `real_res18.pth.tar`
2. For each 3x3 conv layer: extract unique weight groups
3. Cluster steps that share weight groups (horizontal placement)
4. Simulated annealing for routing reduction (vertical placement)
5. Generate LUT-6 INIT truth tables
6. Export all TLMAC hex files

Reference: Gerlinghoff et al., FPGA 2024 | N2UQ: Liu et al., CVPR 2022

In [ ]:
!apt-get install -y git-lfs > /dev/null 2>&1 && git lfs install
!git clone https://github.com/leozqi/resnet18_n2uq.git || true
%cd resnet18_n2uq
!pip install torch torchvision numpy matplotlib 2>/dev/null

In [ ]:
import math, os, json
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. N2UQ Architecture (inline from resnet.py)

In [ ]:
class LearnableBias(nn.Module):
    def __init__(self, out_chn):
        super().__init__()
        self.bias = nn.Parameter(torch.zeros(1, out_chn, 1, 1))
    def forward(self, x):
        return x + self.bias.expand_as(x)

class LTQ(nn.Module):
    """Learnable Threshold Quantiser from N2UQ."""
    def __init__(self, num_bits):
        super().__init__()
        self.n_val = 2 ** num_bits - 1
        self.interval = 2.0 / self.n_val
        self.start = nn.Parameter(torch.tensor([0.0]))
        self.a = nn.Parameter(torch.tensor([self.interval] * self.n_val))
        self.scale1 = nn.Parameter(torch.tensor([1.0]))
        self.scale2 = nn.Parameter(torch.tensor([1.0]))

    def forward(self, x):
        x = x * self.scale1
        a_pos = torch.where(self.a > 1e-3, self.a, torch.tensor(1e-3))
        step_right = torch.tensor(0.0)
        x_f = x
        x_b = x
        for i in range(self.n_val):
            step_right = step_right + self.interval
            if i == 0:
                th_f = self.start + a_pos[0] / 2
                th_b = self.start
                x_f = torch.where(x > th_f, step_right, torch.tensor(0.0))
                x_b = torch.where(x > th_b, self.interval / a_pos[i] * (x - th_b) + step_right - self.interval, torch.tensor(0.0))
            else:
                th_f = th_f + a_pos[i - 1] / 2 + a_pos[i] / 2
                th_b = th_b + a_pos[i - 1]
                x_f = torch.where(x > th_f, step_right, x_f)
                x_b = torch.where(x > th_b, self.interval / a_pos[i] * (x - th_b) + step_right - self.interval, x_b)
        th_b = th_b + a_pos[-1]
        x_b = torch.where(x > th_b, torch.tensor(2.0), x_b)
        return (x_f.detach() + x_b - x_b.detach()) * self.scale2

class HardQuantizeConv(nn.Module):
    """Quantised convolution (HardQuantize) from N2UQ."""
    def __init__(self, in_chn, out_chn, num_bits, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.stride = stride
        self.padding = padding
        self.num_bits = num_bits
        self.kernel_size = kernel_size
        self.clip_val = nn.Parameter(torch.tensor([2.0]))
        self.shape = (out_chn, in_chn, kernel_size, kernel_size)
        self.weight = nn.Parameter((torch.rand(self.shape) - 0.5) * 0.001)

    def forward(self, x):
        real_weights = self.weight
        gamma = (2**self.num_bits - 1) / (2**(self.num_bits - 1))
        sf = gamma * real_weights.abs().mean(dim=[1, 2, 3], keepdim=True)
        sf = sf.detach()
        sw = real_weights / sf
        cw = sw.clamp(-self.clip_val / 2, self.clip_val / 2)
        n = float(2 ** self.num_bits - 1) / self.clip_val
        qw = sf * (torch.round((cw + self.clip_val / 2) * n) / n - self.clip_val / 2)
        return F.conv2d(x, qw, stride=self.stride, padding=self.padding)

    def get_integer_weights(self):
        """Return quantised integer weights for TLMAC extraction."""
        with torch.no_grad():
            rw = self.weight
            gamma = (2**self.num_bits - 1) / (2**(self.num_bits - 1))
            sf = gamma * rw.abs().mean(dim=[1, 2, 3], keepdim=True)
            sw = rw / sf
            cw = sw.clamp(-self.clip_val / 2, self.clip_val / 2)
            n = float(2 ** self.num_bits - 1) / self.clip_val
            qi = torch.round((cw + self.clip_val / 2) * n)  # 0..2^n-1
        return qi.to(torch.int8)

In [ ]:
N_BIT = 3
QUANTIZE_DOWNSAMPLE = True

class BasicBlockQ(nn.Module):
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.bias11 = LearnableBias(inplanes)
        self.prelu1 = nn.PReLU(inplanes)
        self.bias12 = LearnableBias(inplanes)
        self.quan1 = LTQ(N_BIT)
        self.conv1 = HardQuantizeConv(inplanes, planes, N_BIT, 3, stride, 1)
        self.bn1 = nn.BatchNorm2d(planes)
        self.bias21 = LearnableBias(planes)
        self.prelu2 = nn.PReLU(planes)
        self.bias22 = LearnableBias(planes)
        self.quan2 = LTQ(N_BIT)
        self.conv2 = HardQuantizeConv(planes, planes, N_BIT, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.bias31 = LearnableBias(planes)
        self.prelu3 = nn.PReLU(planes)
        self.bias32 = LearnableBias(planes)

    def forward(self, x):
        identity = x
        out = self.bias12(self.prelu1(self.bias11(x)))
        out = self.bn1(self.conv1(self.quan1(out)))
        out = self.bias22(self.prelu2(self.bias21(out)))
        out = self.bn2(self.conv2(self.quan2(out)))
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        out = self.bias32(self.prelu3(self.bias31(out)))
        return out

class N2UQ_ResNet18(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, 2, 1)
        self.layer1 = self._make_block(64, 64, 2)
        self.layer2 = self._make_block(64, 128, 2, stride=2)
        self.layer3 = self._make_block(128, 256, 2, stride=2)
        self.layer4 = self._make_block(256, 512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, 1000)

    def _make_block(self, inplanes, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or inplanes != planes:
            if QUANTIZE_DOWNSAMPLE:
                downsample = nn.Sequential(
                    LTQ(N_BIT),
                    HardQuantizeConv(inplanes, planes, N_BIT, 1, stride, 0),
                    nn.BatchNorm2d(planes))
            else:
                downsample = nn.Sequential(
                    nn.Conv2d(inplanes, planes, 1, stride, bias=False),
                    nn.BatchNorm2d(planes))
        layers = [BasicBlockQ(inplanes, planes, stride, downsample)]
        for _ in range(1, blocks):
            layers.append(BasicBlockQ(planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        x = self.fc(torch.flatten(self.avgpool(x), 1))
        return x

model_n2uq = N2UQ_ResNet18().to(device)

CKPT = 'pretrained_models/quantize_downsampling_true/real_res18.pth.tar'
ckpt = torch.load(CKPT, map_location='cpu')
if isinstance(ckpt, dict) and 'state_dict' in ckpt:
    ckpt = ckpt['state_dict']
model_n2uq.load_state_dict(ckpt, strict=False)
model_n2uq.eval()
for p in model_n2uq.parameters(): p.requires_grad_(False)

print(f'N2UQ ResNet-18 loaded: {CKPT}')
print(f'Parameters: {sum(p.numel() for p in model_n2uq.parameters()):,}')

## 2. Identify 3x3 Conv Layers

In [ ]:
conv_layers = []
for name, module in model_n2uq.named_modules():
    if isinstance(module, HardQuantizeConv) and module.kernel_size in (3, (3, 3)):
        conv_layers.append((name, module))
        print(f'  {name}: {module.shape}  bits={module.num_bits}')
print(f'Total 3x3 conv layers: {len(conv_layers)}')


## 3. TLMAC Parameters & Helpers

G=3 (weight group = one kernel row), BW=3, NCLUS=8 select codes, LUT-6.

In [ ]:
G = 3  # weight group size = D_k for 3x3 conv (paper §3.2)
BW = N_BIT  # weight bit width
BA = N_BIT  # activation bit width (bit-serial iterations)
NCLUS = 2 ** (6 - G)  # 8 select codes per LUT array (paper §3.1.2)
NLUT_PER_ARR = BW + math.ceil(math.log2(G))  # 5 output bits per array
DK = 3
D_P_PER_BLK = 64  # paper §3.2: D_p = 64 * D_k
WEIGHT_OFFSET = (2 ** BW - 1) // 2  # signed weight = qi - offset

print(f'TLMAC: G={G}, BW={BW}, BA={BA}, NCLUS={NCLUS}, '
      f'NLUT_PER_ARR={NLUT_PER_ARR}, WEIGHT_OFFSET={WEIGHT_OFFSET}')
print(f'D_p = 64*D_k = {D_P_PER_BLK * DK} parallel outputs per PE')

def extract_weight_groups(w_q):
    """w_q: [D_o, D_i, 3, 3] unsigned int weights (torch tensor).
    Returns (wg [D_s, D_p] int64 tensor on device, gid_map, D_s, D_p).

    Paper §3.2: D_p = 64*D_k, D_s = D_i * D_o/64.
    Weight tensor [D_o, D_i, D_k, D_k] -> [D_s, D_p, D_k] weight groups,
    where each group is one kernel row of D_k weights.
    """
    D_o, D_i, Dk, _ = w_q.shape
    assert D_o % D_P_PER_BLK == 0, f'D_o={D_o} not divisible by {D_P_PER_BLK}'
    o_blocks = D_o // D_P_PER_BLK
    D_p = D_P_PER_BLK * Dk   # 192
    D_s = D_i * o_blocks      # = D_i * D_o / 64
    # [D_o, D_i, Dk, Dk] -> [D_i, D_o, Dk, Dk] -> [D_i, o_blk, 64, Dk, Dk]
    w = w_q.permute(1, 0, 2, 3).reshape(D_i, o_blocks, D_P_PER_BLK, Dk, Dk)
    # step s = (input_channel, output_block); each step has D_p = 64*Dk groups
    # group = one kernel row: index (o_blk_ch, kernel_row)
    # [D_i, o_blk, 64, Dk(row), Dk(col)] -> [D_i*o_blk, 64, Dk(row), Dk(col)]
    # -> [D_s, 64, Dk(row), Dk(col)] -> take each row as a group of Dk cols
    w = w.reshape(D_s, D_P_PER_BLK, Dk, Dk)        # [D_s, 64, row, col]
    w = w.reshape(D_s, D_p, Dk)                     # [D_s, D_p, G]  (row*64)
    w = w.reshape(-1, G)                            # [D_s*D_p, G]
    unique_groups, inverse = torch.unique(w, dim=0, return_inverse=True)
    wg = inverse.reshape(D_s, D_p).to(torch.int64)
    # gid_map keys are signed integer tuples (for LUT MAC correctness)
    gid_map = {tuple((unique_groups[i] - WEIGHT_OFFSET).tolist()): i
               for i in range(len(unique_groups))}
    return wg, gid_map, D_s, D_p

def spectral_cluster_torch(C_bool, nclus):
    """Spectral clustering + Cluster QR (paper §5.1).
    C_bool: [D_s, N_UWG] bool tensor on device.
    Returns (clusters: list[set[int]], cg: list[set[int]]).
    """
    DS, NG = C_bool.shape
    if DS <= nclus:
        clusters = [{s} for s in range(DS)]
        cg = [{int(g) for g in range(NG) if C_bool[s, g]} for s in range(DS)]
        return clusters, cg

    # Affinity = shared weight groups (C @ C^T), cosine-like
    C_f = C_bool.to(torch.float32)
    A = C_f @ C_f.T                     # [D_s, D_s]
    A.fill_diagonal_(0.0)
    # Normalised symmetric Laplacian L = I - D^{-1/2} A D^{-1/2}
    deg = A.sum(dim=1)
    deg = torch.where(deg > 0, deg, torch.ones_like(deg))
    d_inv_sqrt = deg.rsqrt()
    Lnorm = torch.eye(DS, device=C_bool.device) - (d_inv_sqrt.unsqueeze(1) * A * d_inv_sqrt.unsqueeze(0))
    # Smallest nclus eigenvectors of symmetric matrix = largest of (I - L)
    # Use eigh (DS<=~2048 for ResNet-18 layer4: fine on GPU; CPU fallback below)
    try:
        eigvals, eigvecs = torch.linalg.eigh(Lnorm)   # ascending
    except Exception:
        Lnorm = Lnorm.cpu()
        eigvals, eigvecs = torch.linalg.eigh(Lnorm)
        eigvecs = eigvecs.to(C_bool.device)
    U = eigvecs[:, :nclus]                            # [D_s, nclus]

    # Cluster QR label assignment: QR of U^T, pick pivots, assign by max inner product
    # Row-normalise U (recommended for spectral embedding)
    Un = U / (U.norm(dim=1, keepdim=True) + 1e-12)
    Q, R = torch.linalg.qr(Un.T, mode='reduced')     # Q:[nclus,nclus] R:[nclus,D_s]
    # Pivots = columns of R with largest norm; assign each point to argmax of Un @ Q
    scores = Un @ Q                                   # [D_s, nclus]
    labels = scores.argmax(dim=1)                     # [D_s]

    clusters = [set() for _ in range(nclus)]
    for s in range(DS):
        clusters[int(labels[s].item())].add(s)
    # Drop empty clusters (shouldn't happen, but guard)
    clusters = [c for c in clusters if len(c) > 0]
    cg = []
    for cl in clusters:
        gset = set()
        for s in cl:
            row = C_bool[s]
            idx = torch.nonzero(row, as_tuple=True)[0].tolist()
            gset.update(idx)
        cg.append(gset)
    return clusters, cg

def routing_sa(clusters, wg, n_arr, dp):
    """SA for routing reduction (paper §5.2, Algorithm 1).
    wg: [D_s, D_p] int64 tensor on device (group id per step,output).
    Returns (best_placement: list[{gid:arr}], best_r: int, history: list).

    Route count R = sum_{e,p} 1[exists c: R(e,c,p)!=0], per-cluster.
    Swap move: pick cluster c, arrays e0,e1; swap groups of cluster c
    currently at e0 with those at e1.
    Accept if R_new < R_best or rand < exp((R_best - R_new - 1)/T).
    """
    nclus = len(clusters)
    # Per-cluster: (gid_list tensor, outs_bool [n_g, dp] tensor)
    cg_data = []
    for cl in clusters:
        steps = torch.tensor(sorted(cl), device=wg.device, dtype=torch.long)
        rows = wg[steps]                              # [n_steps, dp]
        # Unique groups in this cluster (1D over the flattened rows) and their output sets
        flat = rows.reshape(-1)                       # [n_steps*dp]
        ug, inv = torch.unique(flat, return_inverse=True)  # ug:[n_g], inv:[n_steps*dp]
        n_g = ug.numel()
        outs = torch.zeros(n_g, dp, dtype=torch.bool, device=wg.device)
        p_idx = torch.arange(dp, device=wg.device).unsqueeze(0).expand(rows.shape).reshape(-1)
        outs[inv, p_idx] = True                       # group g drives output p
        cg_data.append((ug, outs))

    # Placement: assign_c[ci] = tensor[n_g] of array indices
    assign_c = [torch.randint(0, n_arr, (ug.numel(),), device=wg.device) for ug, _ in cg_data]

    def count_routes(assigns):
        # Scatter-OR: used[a[g], p] |= outs[g, p] for each cluster's groups.
        # Uses scatter_reduce_(amax) on float to handle duplicate array indices.
        used = torch.zeros(n_arr, dp, dtype=torch.float32, device=wg.device)
        for ci, (_, outs) in enumerate(cg_data):
            a = assigns[ci]                          # [n_g]
            idx = a.unsqueeze(1).expand_as(outs).long()  # [n_g, dp]
            used.scatter_reduce_(0, idx, outs.float(), reduce='amax')
        return int(used.bool().sum().item())

    def routes_for_cluster(ci, a):
        """Bool [n_arr, dp] of routes contributed by cluster ci."""
        _, outs = cg_data[ci]
        r = torch.zeros(n_arr, dp, dtype=torch.float32, device=wg.device)
        idx = a.unsqueeze(1).expand_as(outs).long()
        r.scatter_reduce_(0, idx, outs.float(), reduce='amax')
        return r.bool()

    curr = [a.clone() for a in assign_c]
    curr_cl_routes = [routes_for_cluster(ci, curr[ci]) for ci in range(nclus)]  # for delta hints
    curr_r = count_routes(curr)
    best = [a.clone() for a in curr]
    best_r = curr_r

    ITERS = int(max(10000, min(200000, 50 * n_arr * dp // 64)))
    ALPHA = 1.4
    hist = []
    for it in range(1, ITERS + 1):
        T = ITERS / (it + 1) ** ALPHA
        c = int(torch.randint(0, nclus, (1,)).item())
        if cg_data[c][0].numel() < 1:
            hist.append((it, curr_r, best_r)); continue
        e0 = int(torch.randint(0, n_arr, (1,)).item())
        e1 = int(torch.randint(0, n_arr, (1,)).item())
        if e0 == e1:
            hist.append((it, curr_r, best_r)); continue
        # Find groups of cluster c at e0 and e1
        a_c = curr[c]
        at_e0 = (a_c == e0).nonzero(as_tuple=True)[0]
        at_e1 = (a_c == e1).nonzero(as_tuple=True)[0]
        # Swap one group between e0 and e1 (if both non-empty) or move one
        cand = []
        if len(at_e0) > 0:
            cand.append((int(at_e0[torch.randint(0, len(at_e0), (1,)).item()].item()), e0, e1))
        if len(at_e1) > 0:
            cand.append((int(at_e1[torch.randint(0, len(at_e1), (1,)).item()].item()), e1, e0))
        if not cand:
            hist.append((it, curr_r, best_r)); continue
        gi, from_e, to_e = cand[int(torch.randint(0, len(cand), (1,)).item())]
        # Neighbour: move one group of cluster c from from_e to to_e.
        # (Algorithm 1 swaps groups between e0/e1; a single-group move is the
        #  atomic swap step when one side has the group and the other receives it.)
        new_assigns = list(curr)
        new_a = a_c.clone()
        new_a[gi] = to_e
        new_assigns[c] = new_a
        # Exact route count (nclus scatter-ORs over [n_arr, dp] bool — cheap)
        nr = count_routes(new_assigns)
        if nr < best_r or torch.rand(1).item() < math.exp((best_r - nr - 1) / max(T, 1e-12)):
            curr = new_assigns
            curr_cl_routes[c] = routes_for_cluster(c, curr[c])
            curr_r = nr
            if nr < best_r:
                best_r = nr
                best = [a.clone() for a in curr]
        hist.append((it, curr_r, best_r))
    # Convert best to list of dicts {gid: arr}
    best_pp = []
    for ci, (ug, _) in enumerate(cg_data):
        best_pp.append({int(g.item()): int(best[ci][i].item()) for i, g in enumerate(ug)})
    return best_pp, best_r, hist


## 4. Compile All Layers

In [ ]:
all_results = {}
wg_assign_all = {}  # layer_name -> wg tensor [D_s, D_p]

for name, module in conv_layers:
    wq = module.get_integer_weights()  # [D_o, D_i, 3, 3] unsigned int8
    # Signed integer weights for LUT MAC (paper: w_g are the quantised weights)
    wq_signed = wq.to(torch.int32) - WEIGHT_OFFSET
    wg, gid_map, D_s, D_p = extract_weight_groups(wq.to(torch.int64))
    N_UWG = len(gid_map)
    wg_assign_all[name] = wg

    # Build C: [D_s, N_UWG] bool — which unique groups each step uses
    C = torch.zeros(D_s, N_UWG, dtype=torch.bool, device=wg.device)
    s_idx = torch.arange(D_s, device=wg.device).unsqueeze(1).expand(-1, D_p).reshape(-1)
    g_idx = wg.reshape(-1)
    C[s_idx, g_idx] = True

    clusters, cg = spectral_cluster_torch(C, NCLUS)
    n_arr = max(len(g) for g in cg) if cg else 1

    best_pp, best_r, hist = routing_sa(clusters, wg, n_arr, D_p)

    # step -> cluster index (= wg_sel code)
    stc = {}
    for ci, cl in enumerate(clusters):
        for s in cl:
            stc[s] = ci

    # --- LUT INIT generation (fully vectorised, one set per array) ---
    id2g = {v: k for k, v in gid_map.items()}
    zero_gid = gid_map.get((0,) * G, 0)

    # Build signed-weight lookup tensor: id2g_tensor[gid] = [w0, w1, w2]
    # Avoids per-entry dict lookup in the wg_all loop.
    id2g_tensor = torch.zeros(N_UWG, G, dtype=torch.int64, device=wg.device)
    for signed_w, gid in gid_map.items():
        for gi in range(G):
            id2g_tensor[gid, gi] = signed_w[gi]

    # Build group_at[ai, sel] = gid via scatter from best_pp.
    # best_pp[ci] = {gid: arr}. We want group_at[arr, ci] = gid.
    # Collect all (arr, ci, gid) triples and scatter.
    ci_list = []; arr_list = []; gid_list = []
    for ci, assign in enumerate(best_pp):
        for gid, ai in assign.items():
            if ai < n_arr:
                ci_list.append(ci); arr_list.append(ai); gid_list.append(gid)
    group_at = torch.full((n_arr, NCLUS), zero_gid, dtype=torch.int64, device=wg.device)
    if ci_list:
        idx = torch.tensor([arr_list, ci_list], device=wg.device).long()  # [2, N]
        vals = torch.tensor(gid_list, device=wg.device, dtype=torch.int64)
        group_at[idx[0], idx[1]] = vals

    # Gather signed weights: wg_all[ai, sel, :] = id2g_tensor[group_at[ai, sel]]
    wg_all = id2g_tensor[group_at]  # [n_arr, NCLUS, G]

    # Vectorised MAC over all 64 addresses
    addr_range = torch.arange(64, device=wg.device)
    sel_all = (addr_range >> G) & (NCLUS - 1)                    # [64]
    abits_all = addr_range & ((1 << G) - 1)                      # [64]
    # act_bits[addr, gi] = bit gi of abits_all[addr]
    gi_range = torch.arange(G, device=wg.device)
    act_bits = ((abits_all.unsqueeze(1) >> gi_range) & 1).to(torch.int64)  # [64, G]
    # sel_wg[ai, addr, gi] = wg_all[ai, sel_all[addr], gi]
    sel_wg = wg_all[:, sel_all.long(), :]                        # [n_arr, 64, G]
    mac = (act_bits.unsqueeze(0) * sel_wg).sum(dim=2)            # [n_arr, 64]
    # 2's complement into N_lut bits (handles negatives)
    mac_u = mac & ((1 << NLUT_PER_ARR) - 1)                      # [n_arr, 64]
    # Pack: init[ai, k] = sum_{addr} bit_k(mac_u[ai,addr]) << addr
    bits = (mac_u.unsqueeze(-1) >> torch.arange(NLUT_PER_ARR, device=wg.device)) & 1  # [n_arr, 64, N_lut]
    addr_mask = (1 << torch.arange(64, device=wg.device))        # [64]
    all_init = (bits * addr_mask.unsqueeze(0).unsqueeze(-1)).sum(dim=1)  # [n_arr, N_lut]

    all_results[name] = dict(
        D_s=D_s, D_p=D_p, N_UWG=N_UWG,
        n_clusters=len(clusters), n_arr=n_arr,
        best_routes=best_r, possible=n_arr * D_p,
        all_init=all_init, step_to_cluster=stc,
        best_placement=best_pp, history=hist,
        id_to_group=id2g, gid_map=gid_map,
        group_at=group_at,
    )
    route_pct = 1.0 - best_r / (n_arr * D_p) if n_arr * D_p > 0 else 0.0
    short = name.replace('.conv', '.c')
    print(f'{short}: D_s={D_s} D_p={D_p} {N_UWG} uwg, {len(clusters)} cl, '
          f'{n_arr} arr, routes {best_r}/{n_arr*D_p} ({route_pct:.0%} red)')

print(f'Compiled {len(all_results)} layers')


## 5. Routing SA Convergence

In [ ]:
n_layers = len(all_results)
if n_layers == 0:
    print('No layers compiled — check previous cell for errors.')
else:
    ncols = min(5, n_layers)
    nrows = (n_layers + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
    if n_layers == 1:
        axes = np.array([axes])
    axes_flat = np.array(axes).flatten() if hasattr(axes, 'flatten') else [axes]
    for idx, (name, res) in enumerate(all_results.items()):
        ax = axes_flat[idx]
        iters = [h[0] for h in res['history']]
        curr  = [h[1] for h in res['history']]
        best  = [h[2] for h in res['history']]
        ax.plot(iters, curr, alpha=0.3, linewidth=0.5)
        ax.plot(iters, best, linewidth=1.5)
        short = name.replace('.conv', '.c')
        ax.set_title(f"{short}\n{res['N_UWG']} grps, {res['n_arr']} arr", fontsize=8)
        ax.tick_params(labelsize=7)
    for idx in range(n_layers, len(axes_flat)):
        axes_flat[idx].set_visible(False)
    plt.tight_layout()
    plt.show()


In [ ]:
total_lut6 = 0
print(f"{'Layer':<30s} {'UWG':>5s} {'Cl':>4s} {'N_arr':>5s} {'LUT-6':>7s} {'Routes':>10s}")
print('-' * 65)
for name, res in all_results.items():
    lut6 = res['n_arr'] * NLUT_PER_ARR  # N_clus groups share N_lut LUTs via select
    total_lut6 += lut6
    short = name.replace('.conv', '.c')
    print(f"{short:<30s} {res['N_UWG']:>5d} {res['n_clusters']:>4d} {res['n_arr']:>5d} {lut6:>7d} {res['best_routes']:>10d}")
print('-' * 65)
print(f'Total LUT-6: {total_lut6:,}')


## 6. Export TLMAC Hex Files

Writes to `tlmac_output/weights/{layer}/`.
Files match the `INIT_FILE` parameter of `lut_array.sv` and `cluster_map_rom.sv`.

In [ ]:
OUT_DIR = 'tlmac_output'
os.makedirs(os.path.join(OUT_DIR, 'weights'), exist_ok=True)
metadata = {}

for name, res in all_results.items():
    safe = name.replace('.', '_')
    ldir = os.path.join(OUT_DIR, 'weights', safe)
    os.makedirs(ldir, exist_ok=True)
    D_s = res['D_s']; D_p = res['D_p']; n_arr = res['n_arr']

    # cluster_map.hex: one entry per step, 1 hex digit (wg_sel, 3 bits for NCLUS=8)
    with open(os.path.join(ldir, 'cluster_map.hex'), 'w') as f:
        for s in range(D_s):
            f.write(f"{res['step_to_cluster'].get(s, 0):X}\n")

    # lut_arr{ai}.hex: one file per LUT array, NLUT_PER_ARR lines of 64-bit INIT.
    # Address = {wg_sel[6-G-1:0], act_bits[G-1:0]} = 6 bits. Encodes all NCLUS
    # weight groups of that array via the select bits (paper §3.1.2, §5.1).
    init = res['all_init'].cpu()  # [n_arr, N_lut]
    for ai in range(n_arr):
        with open(os.path.join(ldir, f'lut_arr{ai}.hex'), 'w') as f:
            for k in range(NLUT_PER_ARR):
                f.write(f"{int(init[ai, k].item()):016X}\n")

    # routing_bitmap.hex: flat [D_p * n_arr] bits, output-major
    # (bit p*n_arr + arr), matching switch_network.sv ROUTING_BITMAP.
    # R[arr, p] = 1 if array arr drives output p (for some cluster).
    wg = wg_assign_all[name]
    R = torch.zeros(n_arr, D_p, dtype=torch.bool, device=wg.device)
    for ci, assign in enumerate(res['best_placement']):
        # group -> arr for this cluster; group -> outputs globally
        for gid, ai in assign.items():
            # outputs p where any step in cluster ci has wg[s,p]==gid
            steps = [s for s in res['step_to_cluster'] if res['step_to_cluster'][s] == ci]
            st = torch.tensor(steps, device=wg.device, dtype=torch.long)
            sub = wg[st]  # [n_steps, D_p]
            mask = (sub == gid).any(dim=0)  # [D_p]
            R[ai] |= mask
    # Flatten output-major: bit index = p * n_arr + arr
    R_flat = R.t().reshape(-1)  # [D_p * n_arr]
    n_bits = D_p * n_arr
    n_words = (n_bits + 63) // 64
    with open(os.path.join(ldir, 'routing_bitmap.hex'), 'w') as f:
        for wi in range(n_words):
            word = 0
            for b in range(64):
                idx = wi * 64 + b
                if idx < n_bits and bool(R_flat[idx].item()):
                    word |= (1 << b)
            f.write(f"{word:016X}\n")

    metadata[safe] = {
        'layer': name, 'D_s': D_s, 'D_p': D_p, 'D_o_blocks': D_s // (D_p // DK),
        'N_UWG': res['N_UWG'], 'clusters': res['n_clusters'],
        'n_arr': n_arr, 'G': G, 'BW': BW, 'BA': BA,
        'NLUT_PER_ARR': NLUT_PER_ARR, 'NCLUS': NCLUS,
        'WEIGHT_OFFSET': WEIGHT_OFFSET,
        'best_routes': res['best_routes'], 'possible': res['possible'],
        'routing_bitmap_order': 'output_major',  # bit = p*n_arr + arr
        'routing_bitmap_bits': n_bits,
    }

with open(os.path.join(OUT_DIR, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Exported {len(metadata)} layers to {OUT_DIR}/')
for k, v in metadata.items():
    print(f"  {k}: D_s={v['D_s']} D_p={v['D_p']} {v['n_arr']} arr, "
          f"{v['clusters']} cl, {v['best_routes']} routes")

for root, dirs, files in os.walk(os.path.join(OUT_DIR, 'weights')):
    for fname in sorted(files):
        sz = os.path.getsize(os.path.join(root, fname))
        rel = os.path.relpath(os.path.join(root, fname), OUT_DIR)
        print(f'  {rel}  ({sz:,} bytes)')


In [ ]:
print('\n=== TLMAC Integrated Compilation Summary ===')
print(f'Model: N2UQ ResNet-18 ({N_BIT}-bit, quantize_downsample={QUANTIZE_DOWNSAMPLE})')
print(f'Layers compiled:       {len(all_results)}')
total_groups = sum(r['N_UWG'] for r in all_results.values())
print(f'Total unique groups:   {total_groups:,}')
total_arr = sum(r['n_arr'] for r in all_results.values())
print(f'Total LUT arrays:      {total_arr}')
print(f'Total LUT-6:           {total_lut6:,}')
print(f'Output:                {OUT_DIR}/')

In [ ]:
# --- Download TLMAC outputs as .tar.gz ---
import tarfile
from google.colab import files

ARCHIVE = 'tlmac_output.tar.gz'
with tarfile.open(ARCHIVE, 'w:gz') as tar:
    tar.add(OUT_DIR, arcname='tlmac_output')

archive_size = os.path.getsize(ARCHIVE)
print(f'Created {ARCHIVE} ({archive_size:,} bytes)')

# List contents
with tarfile.open(ARCHIVE, 'r:gz') as tar:
    for m in tar.getmembers():
        print(f'  {m.name}  ({m.size:,} bytes)')

files.download(ARCHIVE)
